# LangChain Tool-Execution Loop

This notebook demonstrates the **tool-execution loop** — the core mechanic behind LangChain agents.

Instead of answering from its training data, the LLM is given a `get_weather` tool. Each time it decides it needs weather data it emits a **tool call**; we execute the tool and feed the result back; the LLM then writes its final answer. This think → act → observe cycle repeats until the LLM signals it is done.

| Step | Who acts | What happens |
|---|---|---|
| 1 | Human | Sends a question |
| 2 | LLM | Reads question, emits a `tool_call` |
| 3 | Python | Executes the tool, gets the result |
| 4 | LLM | Reads the result, writes the final answer |

**Provider:** Anthropic Claude (`claude-haiku-4-5-20251001`) via `langchain-anthropic`.

### Step 1 — Load Environment Variables

**What:** Reads `ANTHROPIC_API_KEY` from the `.env` file.

**Why:** API keys must never be hardcoded. `python-dotenv` reads them at runtime from a local `.env` file so the secret stays out of the notebook.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
print("ANTHROPIC_API_KEY:", bool(ANTHROPIC_API_KEY))

ANTHROPIC_API_KEY: True


---
### Step 2 — Define the `get_weather` Tool

**What:** Declares a plain Python function and wraps it with LangChain's `@tool` decorator to turn it into a tool the LLM can call.

**Why the `@tool` decorator matters:** It automatically extracts three things the LLM needs to decide *when* and *how* to call the tool:
1. **Name** — `get_weather` (taken from the function name).
2. **Description** — the docstring. The LLM reads this to understand what the tool does.
3. **Input schema** — the function's type-annotated parameters (`city: str`). LangChain converts this to a JSON Schema so the LLM knows what arguments to pass.

**Why a fake `weather_db`:** Simulates a real weather API without an external HTTP call. Replace with OpenWeatherMap etc. in production.

**The sanity-check `print`:** Calls the tool *directly* (bypassing the LLM) to confirm it works before wiring it into the loop.

In [3]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Return the current weather for a given city name."""
    weather_db = {
        "dhaka":      {"temperature": "34°C", "condition": "Sunny",         "humidity": "72%", "wind_speed": "10 km/h", "feels_like": "38°C"},
        "chittagong": {"temperature": "32°C", "condition": "Partly Cloudy", "humidity": "78%", "wind_speed": "14 km/h", "feels_like": "36°C"},
        "london":     {"temperature": "17°C", "condition": "Overcast",      "humidity": "85%", "wind_speed": "20 km/h", "feels_like": "15°C"},
    }

    key = city.lower().strip()
    if key not in weather_db:
        return f"No weather data available for '{city}'."

    w = weather_db[key]
    return (
        f"Weather in {city.title()}:\n"
        f"  Temperature : {w['temperature']} (feels like {w['feels_like']})\n"
        f"  Condition   : {w['condition']}\n"
        f"  Humidity    : {w['humidity']}\n"
        f"  Wind Speed  : {w['wind_speed']}"
    )


# Sanity check — call the tool directly, no LLM involved
print(get_weather.invoke("Dhaka"))

Weather in Dhaka:
  Temperature : 34°C (feels like 38°C)
  Condition   : Sunny
  Humidity    : 72%
  Wind Speed  : 10 km/h


---
### Step 3 — Bind the Tool to the LLM

**What:** Creates a `ChatAnthropic` instance and binds `get_weather` to it with `.bind_tools()`.

**Why `.bind_tools()`:** Attaches the tool's JSON schema to every request. Without this the LLM has no knowledge the tool exists and answers from training data only.

**Two response modes after binding:**
- **Text mode** — `AIMessage` with `.content` set (used for the final answer).
- **Tool-call mode** — `AIMessage` where `.content` is empty but `.tool_calls` is a list of `{name, args, id}` dicts.

The loop in Step 4 checks which mode the model chose and acts accordingly.

In [6]:
from langchain_anthropic import ChatAnthropic

tools = [get_weather]

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=ANTHROPIC_API_KEY,
)

llm_with_tools = llm.bind_tools([get_weather])

print("Tool bound:", [t.name for t in tools])

Tool bound: ['get_weather']


---
### Step 4 — Build the Tool-Execution Loop

**What:** A manual for-loop that implements the think → act → observe cycle without a high-level agent abstraction.

**Why build it manually:** Seeing the loop explicitly makes the mechanics transparent. High-level agent APIs (LangGraph, `create_agent`) do exactly this under the hood.

**Each iteration:**
1. **Think** — `llm_with_tools.invoke(messages)`. The full conversation history is passed in so the model has context from all prior turns.
2. **Check** — if `response.tool_calls` is empty the LLM produced a final text answer; exit the loop.
3. **Act** — for each tool call: dispatch to the matching Python function via `tools_map`, collect the result.
4. **Observe** — wrap the result in a `ToolMessage`, append it with the `AIMessage` to `messages`, loop again.

**`ToolMessage` fields:**
- `tool_call_id` — links the result to the specific tool call that requested it (critical when multiple tools run in one turn).
- `content` — the string the tool returned.

**`max_iterations` guard:** Prevents infinite loops if the model keeps emitting tool calls without converging on a final answer.

In [7]:
from langchain_core.messages import HumanMessage, ToolMessage

# Map tool names to Python callables for dispatch
tools_map = {"get_weather": get_weather}

def run_tool_loop(user_question: str, max_iterations: int = 5) -> str:
    messages = [HumanMessage(content=user_question)]

    for iteration in range(1, max_iterations + 1):
        print(f"[Iteration {iteration}] Calling LLM...")
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # No tool calls → LLM produced a final text answer
        if not response.tool_calls:
            print(f"[Iteration {iteration}] Final answer received.")
            return response.content

        # Execute every tool the LLM requested this turn
        for tc in response.tool_calls:
            print(f"[Iteration {iteration}] Tool call  -> {tc['name']}({tc['args']})")
            result = tools_map[tc["name"]].invoke(tc["args"])
            print(f"[Iteration {iteration}] Tool result-> {result}")
            messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))

    return "[Max iterations reached without a final answer]"


answer = run_tool_loop("What is the weather like in Dhaka right now?")
print("\n=== Final Answer ===")
print(answer)

[Iteration 1] Calling LLM...
[Iteration 1] Tool call  -> get_weather({'city': 'Dhaka'})
[Iteration 1] Tool result-> Weather in Dhaka:
  Temperature : 34°C (feels like 38°C)
  Condition   : Sunny
  Humidity    : 72%
  Wind Speed  : 10 km/h
[Iteration 2] Calling LLM...
[Iteration 2] Final answer received.

=== Final Answer ===
The weather in Dhaka right now is:

- **Temperature:** 34°C (feels like 38°C due to humidity)
- **Condition:** Sunny
- **Humidity:** 72%
- **Wind Speed:** 10 km/h

It's quite warm and humid, with sunny conditions. The heat index makes it feel warmer than the actual temperature, so it's a good idea to stay hydrated and use sun protection if you're going outside.


---
### Step 5 — Multiple Cities (Parallel Tool Calls)

**What:** One question that requires two city lookups, showing the LLM can emit **multiple tool calls in one turn**.

**Why this matters:** Instead of calling tools one-by-one the LLM batches both into a single response. The loop handles this naturally because `response.tool_calls` is a list — both are executed before the next LLM turn.

**Expected flow:**
1. LLM emits `get_weather("London")` and `get_weather("Chittagong")` in the same response.
2. Both are executed; two `ToolMessage` objects are appended.
3. LLM reads both results and writes a single comparison answer.

In [8]:
answer = run_tool_loop(
    "Compare the weather in London and Chittagong. Which city is warmer?"
)
print("\n=== Final Answer ===")
print(answer)

[Iteration 1] Calling LLM...
[Iteration 1] Tool call  -> get_weather({'city': 'London'})
[Iteration 1] Tool result-> Weather in London:
  Temperature : 17°C (feels like 15°C)
  Condition   : Overcast
  Humidity    : 85%
  Wind Speed  : 20 km/h
[Iteration 1] Tool call  -> get_weather({'city': 'Chittagong'})
[Iteration 1] Tool result-> Weather in Chittagong:
  Temperature : 32°C (feels like 36°C)
  Condition   : Partly Cloudy
  Humidity    : 78%
  Wind Speed  : 14 km/h
[Iteration 2] Calling LLM...
[Iteration 2] Final answer received.

=== Final Answer ===
**Weather Comparison:**

**London:**
- Temperature: 17°C (feels like 15°C)
- Condition: Overcast
- Humidity: 85%
- Wind Speed: 20 km/h

**Chittagong:**
- Temperature: 32°C (feels like 36°C)
- Condition: Partly Cloudy
- Humidity: 78%
- Wind Speed: 14 km/h

**Answer: Chittagong is significantly warmer.** It's 15°C hotter than London (32°C vs 17°C). In fact, Chittagong feels even warmer at 36°C due to the higher humidity and feels-like tem

---
### Step 6 — Graceful Handling of Unknown Cities

**What:** Asks for a city not in `weather_db` to show how missing data flows back to the LLM.

**Why this matters:** The LLM still calls the tool (it does not know which cities are available). The tool returns `"No weather data available"`. The LLM reads that `ToolMessage` and writes an honest reply — it does **not** hallucinate a weather report. This is the key advantage of tool-grounded responses over ungrounded LLM answers.

In [ ]:
answer = run_tool_loop("What's the weather in Tokyo?")
print("\n=== Final Answer ===")
print(answer)

[Iteration 1] Calling LLM...
[Iteration 1] Tool call  -> get_weather({'city': 'Tokyo'})
[Iteration 1] Tool result-> No weather data available for 'Tokyo'.
[Iteration 2] Calling LLM...
[Iteration 2] Final answer received.

=== Final Answer ===
I apologize, but the weather service doesn't have current weather data available for Tokyo at the moment. This could be due to a temporary service issue or limited data availability. 

You might want to try:
- Checking a weather website directly like Weather.com, AccuWeather, or the Japan Meteorological Corporation's website
- Looking up weather for a different city
- Trying again in a few moments

Is there anything else I can help you with?


: 